# Aula 6: montando o seu banco de fraude em cartão no FreeSQL

Este notebook cria as mesmas 6 tabelas usadas na aula, dentro do **seu** schema do FreeSQL, com os dados que a coordenação preparou. Rode as células na ordem, de cima para baixo.

**Antes de começar:**
- Tenha em mãos o DSN, o usuário e a senha do seu schema, do painel "Schema Connection Details" do FreeSQL.
- Os arquivos `schema.sql` e a pasta `csv/` precisam estar na mesma pasta do Google Drive que este notebook. Se você copiou a pasta inteira, já está certo.

## 1. Instalar o driver do Oracle

In [ ]:
!pip install oracledb -q

## 2. Montar o seu Google Drive

O Colab vai pedir para você autorizar o acesso.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Apontar para a pasta com os dados

Ajuste o caminho abaixo para onde esta pasta ficou dentro do seu Drive.

In [ ]:
PASTA_DADOS = "/content/drive/MyDrive/AI Talent Academy/Aula 06/dados"

## 4. Conectar no seu FreeSQL

Preencha com os dados do painel "Schema Connection Details" da **sua** conta. A senha fica escondida enquanto você digita.

In [ ]:
import getpass
import oracledb

dsn = input("DSN (exemplo: tcps://db.freesql.com:2484/SEU_SERVICO): ")
usuario = input("Usuário: ")
senha = getpass.getpass("Senha: ")

conexao = oracledb.connect(user=usuario, password=senha, dsn=dsn)
print("Conectado.")

## 5. Criar as 6 tabelas

Se você já rodou este notebook antes e quer recomeçar do zero, rode a célula de `DROP TABLE` logo abaixo primeiro. Se é a primeira vez, pule direto para a célula de `CREATE TABLE`.

In [ ]:
# Rode esta célula só se quiser apagar tabelas de uma tentativa anterior.
cursor = conexao.cursor()
ordem_para_apagar = ['contestacoes', 'transacoes', 'cartoes', 'lojistas', 'clientes', 'categorias']
for tabela in ordem_para_apagar:
    try:
        cursor.execute(f'DROP TABLE {tabela}')
    except oracledb.DatabaseError as erro:
        codigo = erro.args[0].code
        if codigo != 942:  # 942 = tabela nao existe, pode ignorar
            raise
conexao.commit()
print("Tabelas antigas removidas, se havia alguma.")

In [ ]:
import pathlib

schema_sql = pathlib.Path(f"{PASTA_DADOS}/schema.sql").read_text(encoding="utf-8")
cursor = conexao.cursor()
for instrucao in [s.strip() for s in schema_sql.split(';') if s.strip()]:
    cursor.execute(instrucao)
conexao.commit()
print("As 6 tabelas foram criadas.")

## 6. Carregar os dados dos arquivos CSV

Cada tabela vira uma função que lê o CSV correspondente e insere em lotes de 2 mil linhas, na ordem certa por causa das chaves estrangeiras.

In [ ]:
import csv
from datetime import datetime

# Para cada tabela, o tipo de cada coluna que não é texto puro.
# 'data' vira datetime ou date, int e float viram numero de verdade.
CONVERSORES = {
    "categorias": {"id_categoria": int},
    "clientes": {"id_cliente": int, "data_nascimento": "data",
                 "data_cadastro": "data", "score_risco": float},
    "lojistas": {"id_lojista": int, "id_categoria": int, "data_cadastro": "data"},
    "cartoes": {"id_cartao": int, "id_cliente": int, "data_emissao": "data",
                "limite": float},
    "transacoes": {"id_transacao": int, "id_cartao": int, "id_lojista": int,
                   "valor": float, "data_hora": "data"},
    "contestacoes": {"id_contestacao": int, "id_transacao": int,
                     "data_contestacao": "data"},
}


def converter_data(valor):
    try:
        return datetime.strptime(valor, "%Y-%m-%d %H:%M:%S")
    except ValueError:
        return datetime.strptime(valor, "%Y-%m-%d").date()


def converter_valor(valor, conversor):
    if valor == "":
        return None
    if conversor == "data":
        return converter_data(valor)
    if conversor in (int, float):
        return conversor(valor)
    return valor


def carregar_csv(nome_tabela, tamanho_lote=2000):
    caminho = f"{PASTA_DADOS}/csv/{nome_tabela}.csv"
    conversores = CONVERSORES.get(nome_tabela, {})
    with open(caminho, encoding="utf-8") as f:
        leitor = csv.reader(f)
        colunas = next(leitor)
        linhas = []
        for linha_bruta in leitor:
            linha = [converter_valor(v, conversores.get(col))
                     for v, col in zip(linha_bruta, colunas)]
            linhas.append(tuple(linha))

    marcadores = ", ".join(f":{i + 1}" for i in range(len(colunas)))
    sql = f"INSERT INTO {nome_tabela} ({', '.join(colunas)}) VALUES ({marcadores})"
    total = len(linhas)
    for inicio in range(0, total, tamanho_lote):
        lote = linhas[inicio:inicio + tamanho_lote]
        cursor.executemany(sql, lote)
        conexao.commit()
    print(f"{nome_tabela}: {total} linhas carregadas.")

In [ ]:
for tabela in ['categorias', 'clientes', 'lojistas', 'cartoes', 'transacoes', 'contestacoes']:
    carregar_csv(tabela)

## 7. Conferir se bateu

In [ ]:
for tabela in ['categorias', 'clientes', 'lojistas', 'cartoes', 'transacoes', 'contestacoes']:
    cursor.execute(f'SELECT COUNT(*) FROM {tabela}')
    (total,) = cursor.fetchone()
    print(f'{tabela}: {total} linhas')

## Pronto

O seu banco está populado, com o mesmo caso de fraude em cartão usado nas Aulas 2 e 5, agora em 6 tabelas e volume de verdade. A partir daqui, use o worksheet do FreeSQL ou continue neste mesmo notebook para rodar as consultas da aula.